# Инференс и проверка ответа

Сначала запустите `solution.ipynb`: он создаст артефакты в `notebook_outputs/`.
Затем выполните этот ноутбук в той же папке. Нужны только benchmark Parquet,
сохранённые артефакты и локальные библиотеки; внешние API не используются.
Результат — `answer.csv` с двумя колонками, проверенный после записи.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'scipy': 'scipy', 'sklearn': 'scikit-learn', 'nltk': 'nltk',
    'tqdm': 'tqdm', 'cloudpickle': 'cloudpickle',
    'sentence_transformers': 'sentence-transformers',
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])


In [ ]:
from pathlib import Path
import gzip
import hashlib
import json

import cloudpickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer
from IPython.display import display

DATA_DIR = Path('datasets')
if not (DATA_DIR / 'benchmark_queries.parquet').is_file() and \
        Path('benchmark_queries.parquet').is_file():
    DATA_DIR = Path('.')
REPORT_DIR = Path('notebook_outputs')
ANSWER_PATH = Path('answer.csv')
queries = pq.read_table(DATA_DIR / 'benchmark_queries.parquet').to_pandas()
corpus_ids = pq.read_table(DATA_DIR / 'benchmark_items.parquet',
                           columns=['item_id']).column('item_id').to_pylist()

with gzip.open(REPORT_DIR / 'model.pkl.gz', 'rb') as stream:
    model = cloudpickle.load(stream)
fusion = json.loads((REPORT_DIR / 'fusion.json').read_text())
document_vectors = np.load(REPORT_DIR / 'e5_item_vectors.npy')
item_order_hash = hashlib.sha256('\n'.join(model.item_ids).encode()).hexdigest()
assert item_order_hash == fusion['item_order_sha256']
assert document_vectors.shape[0] == len(model.item_ids) == len(corpus_ids)
assert set(model.item_ids) == set(corpus_ids)

DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
          else 'cpu')
encoder = SentenceTransformer(str(REPORT_DIR / 'e5_encoder'), device=DEVICE)
encoder.max_seq_length = fusion['max_seq_length']
print('Запросов:', len(queries), 'объявлений:', len(model.item_ids),
      'устройство:', DEVICE)


In [ ]:
import numpy as np
import torch

def topk(scores: np.ndarray, k: int) -> np.ndarray:
    """Детерминированный top-k: при равных оценках меньший индекс раньше.

    Документы заранее сортируются по item_id. Учитываем также равенство
    оценок на границе top-k, которое один argpartition не разрешает стабильно.
    """
    k = min(k, len(scores))
    if k == 0:
        return np.empty(0, dtype=np.int32)
    cutoff = np.partition(scores, len(scores) - k)[len(scores) - k]
    above = np.flatnonzero(scores > cutoff)
    tied = np.flatnonzero(scores == cutoff)[:k - len(above)]
    selected = np.concatenate([above, tied])
    return selected[np.lexsort((selected, -scores[selected]))].astype(np.int32)

def query_texts(queries):
    texts = queries.search_query.fillna('').astype(str).tolist()
    filters = queries.search_infm_params_text.fillna('').astype(str).tolist()
    return [f'query: {text}. {params}' for text, params in zip(texts, filters)]

def similarity_batches(encoder, document_vectors, queries, device, batch_size=32):
    vectors = encoder.encode(query_texts(queries), batch_size=64, device=device,
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=False).astype(np.float16)
    docs = torch.from_numpy(np.asarray(document_vectors)).to(device)
    for start in range(0, len(vectors), batch_size):
        stop = min(start + batch_size, len(vectors))
        q = torch.from_numpy(vectors[start:stop]).to(device)
        # Одинаковая точность при валидации и финальном инференсе.
        yield start, (q @ docs.T).float().cpu().numpy().astype(np.float16).astype(np.float32)

def hybrid_channels(model, raw, similarity):
    lexical = model.combine(raw, model.config)
    lexical = lexical / max(float(lexical.max()), 1e-6)
    similarity = np.asarray(similarity, dtype=np.float32)
    median = float(np.median(similarity))
    upper = float(np.quantile(similarity, 0.999))
    semantic = np.clip((similarity - median) / max(upper - median, 1e-6),
                       0, 1.4)
    geo = (1 + 10 * raw[:, 4]) / 11
    category = (1 + 0.7 * raw[:, 5]) / 1.7
    semantic *= geo * category
    return lexical, semantic

def hybrid_topk(lexical, semantic, rule, topk, k=50):
    method, value = rule.split('_', 1)
    weight = float(value)
    if method == 'blend':
        score = (1 - weight) * lexical + weight * semantic
    elif method == 'rrf':
        a = topk(lexical, 1000)
        b = topk(semantic, 1000)
        score = np.zeros(len(lexical), dtype=np.float32)
        score[a] = 1 / (60 + np.arange(len(a)))
        score[b] += weight / (60 + np.arange(len(b)))
    else:
        raise ValueError(f'Неизвестное правило объединения: {rule}')
    return topk(score, k)

def hybrid_predict(model, encoder, document_vectors, queries, device, topk, rule):
    predictions = []
    features = model.iter_features(queries, progress=False)
    for start, batch in similarity_batches(encoder, document_vectors, queries, device):
        for similarity in batch:
            raw = next(features)
            lexical, semantic = hybrid_channels(model, raw, similarity)
            positions = hybrid_topk(lexical, semantic, rule, topk)
            predictions.append(model.item_ids[positions].tolist())
        print(f'Запросы: {start + len(batch)}/{len(queries)}', end='\r', flush=True)
    print()
    return predictions

In [ ]:
"""Строгая проверка формата сдачи, включая чтение сохранённого CSV."""

import re
from pathlib import Path

import pandas as pd


def validate_submission(answer, queries, corpus_ids):
    if list(answer.columns) != ["query_id", "answer"]:
        raise ValueError(
            "Ожидаются ровно колонки query_id,answer в таком порядке")
    expected = queries.query_id.tolist()
    if len(expected) != len(set(expected)):
        raise ValueError("Повтор query_id во входных запросах")
    if len(answer) != len(expected) or answer.query_id.duplicated().any():
        raise ValueError("Неверное число строк или повторы query_id")
    if set(answer.query_id) != set(expected):
        raise ValueError("Множество query_id не совпадает с benchmark_queries")
    corpus_ids = set(corpus_ids)
    lengths = []
    for qid, raw in answer.itertuples(index=False, name=None):
        if not isinstance(qid, str) or len(qid) != 16:
            raise ValueError(f"Неверный query_id: {qid!r}")
        if not isinstance(raw, str):
            raise ValueError("answer должен быть строкой")
        ids = raw.split(" ") if raw else []
        if not 1 <= len(ids) <= 50 or len(ids) != len(set(ids)):
            raise ValueError(f"Неверное число/повторы item_id для {qid}")
        if any(not re.fullmatch(r"[0-9a-f]{16}", i) for i in ids):
            raise ValueError(f"Неверный формат item_id для {qid}")
        if not set(ids) <= corpus_ids:
            raise ValueError(f"item_id за пределами корпуса для {qid}")
        lengths.append(len(ids))
    return {"rows": len(answer), "min_candidates": min(lengths),
            "max_candidates": max(lengths), "valid": True}


def save_submission(queries, predictions, corpus_ids, output):
    answer = pd.DataFrame({"query_id": queries.query_id,
                           "answer": [" ".join(ids) for ids in predictions]})
    validate_submission(answer, queries, corpus_ids)
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    answer.to_csv(output, index=False, encoding="utf-8", lineterminator="\n")
    reread = pd.read_csv(output, dtype=str, keep_default_na=False)
    return validate_submission(reread, queries, corpus_ids)


In [ ]:
predictions = hybrid_predict(model, encoder, document_vectors,
                             queries, DEVICE, topk, fusion['rule'])
checks = save_submission(queries, predictions, corpus_ids, ANSWER_PATH)
print('Проверка:', checks)
print('SHA256:', hashlib.sha256(ANSWER_PATH.read_bytes()).hexdigest())
print('Файл:', ANSWER_PATH.resolve())
display(pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False).head())
